## Классификация: превышает ли значение IC50 медианное значение выборки

In [1]:
import pandas as pd

import sys
sys.path.append('../src')
from preprocessing import DatasetPreprocessor
from metrics_calculator import MetricsCalculator

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

In [2]:
df = pd.read_excel('../data/raw/classicMLCourseWorkData.xlsx')

# удаляем пропуски
df = df.dropna().copy()

# медиана таргета
ic50_median = df['IC50, mM'].median()

# бинарный таргет
y = (df['IC50, mM'] > ic50_median).astype(int)

# признаки
X = df.drop(columns=['IC50, mM'])

In [3]:
# Класс расчета метрик
metrics_helper = MetricsCalculator()

In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

#### Логистическая регрессия

In [5]:
logreg_pipeline = Pipeline([
    (
        'preprocessor',
        DatasetPreprocessor(
            target_column=None,
            drop_unnamed=True,
            drop_leakage=True,
            drop_constant_features=True,
            drop_corr=True,
            corr_threshold=0.9,
            log_features=True,
            log_skew_threshold=2.0,
            log_min_unique_values=10
        )
    ),
    ('scaler', StandardScaler()),
    (
        'model',
        LogisticRegression(
            max_iter=1000,
            random_state=42
        )
    )
])

# обучение
logreg_pipeline.fit(X_train, y_train)

# предсказания
logreg_y_train_pred = logreg_pipeline.predict(X_train)
logreg_y_test_pred = logreg_pipeline.predict(X_test)


logreg_train_metrics = metrics_helper.classification_metrics(y_train, logreg_y_train_pred)
logreg_test_metrics = metrics_helper.classification_metrics(y_test, logreg_y_test_pred)

print('Метрики на train:')
display(logreg_train_metrics)

print('Метрики на test:')
display(logreg_test_metrics)

Метрики на train:


{'accuracy': 0.843358395989975,
 'precision': 0.8309178743961353,
 'recall': 0.8621553884711779,
 'f1': 0.8462484624846248}

Метрики на test:


{'accuracy': 0.785,
 'precision': 0.7614678899082569,
 'recall': 0.83,
 'f1': 0.7942583732057417}

- Модель показывает хорошее качество на train (accuracy $\approx 0.84$), на test качество немного ниже (accuracy $\approx 0.79$)
- Метрики precision, recall и f1-score находятся на схожем уровне
- Устойчивое качество без сильного переобучения

**Вывод:** логистическая регрессия даёт адекватный baseline для текущей задачи

#### Случайный лес

In [6]:
# pipeline
rf_pipeline = Pipeline([
    (
        'preprocessor',
        DatasetPreprocessor(
            target_column=None,
            drop_unnamed=True,
            drop_leakage=True,
            drop_constant_features=True,
            drop_corr=True,
            corr_threshold=0.9,
            log_features=True,
            log_skew_threshold=2.0,
            log_min_unique_values=10
        )
    ),
    ('scaler', StandardScaler()),
    (
        'model',
        RandomForestClassifier(
            n_estimators=100,
            max_depth=None,
            random_state=42,
            n_jobs=-1
        )
    )
])

# обучение
rf_pipeline.fit(X_train, y_train)

# предсказания
rf_y_train_pred = rf_pipeline.predict(X_train)
rf_y_test_pred = rf_pipeline.predict(X_test)

# метрики
rf_train_metrics = metrics_helper.classification_metrics(y_train, rf_y_train_pred)
rf_test_metrics = metrics_helper.classification_metrics(y_test, rf_y_test_pred)

print('Метрики на train:')
display(rf_train_metrics)

print('Метрики на test:')
display(rf_test_metrics)

Метрики на train:


{'accuracy': 1.0, 'precision': 1.0, 'recall': 1.0, 'f1': 1.0}

Метрики на test:


{'accuracy': 0.885,
 'precision': 0.8811881188118812,
 'recall': 0.89,
 'f1': 0.8855721393034826}

- Модель показывает идеальное качество на train (accuracy = 1.0), что указывает на переобучение
- На test качество остаётся высоким (accuracy $\approx 0.89$), значительно выше baseline
- Метрики precision, recall и f1-score находятся на высоком уровне

**Вывод:** RandomForest значительно улучшает качество по сравнению с логистической регрессией, несмотря на переобучение, и является более подходящей моделью для задачи классификации IC50

In [7]:
# сетка гиперпараметров
rf_param_grid = {
    'model__n_estimators': [100, 200],
    'model__max_depth': [5, 10, None],
    'model__min_samples_split': [2, 5],
    'model__min_samples_leaf': [1, 2],
    'model__max_features': ['sqrt', 0.5]
}

# GridSearch
rf_grid_search = GridSearchCV(
    estimator=rf_pipeline,
    param_grid=rf_param_grid,
    cv=5,
    scoring='f1',
    n_jobs=-1
)

# обучение
rf_grid_search.fit(X_train, y_train)

# лучшая модель
best_rf_pipeline = rf_grid_search.best_estimator_

print('Лучшие параметры:')
print(rf_grid_search.best_params_)

# предсказания
best_rf_y_train_pred = best_rf_pipeline.predict(X_train)
best_rf_y_test_pred = best_rf_pipeline.predict(X_test)

# метрики
best_rf_train_metrics = metrics_helper.classification_metrics(y_train, best_rf_y_train_pred)
best_rf_test_metrics = metrics_helper.classification_metrics(y_test, best_rf_y_test_pred)

print('Метрики на train:')
display(best_rf_train_metrics)

print('Метрики на test:')
display(best_rf_test_metrics)

Лучшие параметры:
{'model__max_depth': 10, 'model__max_features': 0.5, 'model__min_samples_leaf': 2, 'model__min_samples_split': 2, 'model__n_estimators': 100}
Метрики на train:


{'accuracy': 0.9949874686716792,
 'precision': 1.0,
 'recall': 0.9899749373433584,
 'f1': 0.9949622166246851}

Метрики на test:


{'accuracy': 0.975,
 'precision': 0.9797979797979798,
 'recall': 0.97,
 'f1': 0.9748743718592965}

- Модель показывает почти идеальное качество на train (accuracy $\approx 0.99$) и очень высокое качество на test (accuracy $\approx 0.98$)
- Разрыв между train и test минимален, переобучение практически отсутствует
- Метрики precision, recall и f1-score на test высокие


In [9]:
best_rf_y_test_proba = best_rf_pipeline.predict_proba(X_test)[:, 1]
best_rf_full_test_metrics = metrics_helper.full_classification_metrics(
    y_true=y_test,
    y_pred=best_rf_y_test_pred,
    y_proba=best_rf_y_test_proba
)

print('Полный набор метрик на test:')
display(best_rf_full_test_metrics)

Полный набор метрик на test:


{'accuracy': 0.975,
 'precision': 0.9797979797979798,
 'recall': 0.97,
 'f1': 0.9748743718592965,
 'roc_auc': 0.9973,
 'tn': np.int64(98),
 'fp': np.int64(2),
 'fn': np.int64(3),
 'tp': np.int64(97)}

- Модель показывает очень высокое качество на test (accuracy $\approx 0.98$, f1 $\approx 0.97$)
- ROC-AUC $\approx 0.997$, что говорит о практически идеальном разделении классов
- Ошибки минимальны: FP = 2, FN = 3, модель стабильно классифицирует оба класса

**Вывод:** RandomForest показывает высокое и  качество без существенного переобучения и является финальным решением для задачи классификации IC50